# 05 — Mở giao diện Streamlit giống web demo

Notebook khôi phục đúng `ui_app.py` và output mới nhất, khởi động Streamlit trong
máy ảo Colab rồi tạo một đường dẫn TryCloudflare tạm thời. Không cần tài khoản
Cloudflare hoặc token. Hãy giữ phiên Colab hoạt động trong lúc trình bày.

> Đường dẫn là công khai và tạm thời; chỉ chia sẻ trong buổi demo.

In [ ]:
#@title Clone dự án từ GitHub và cài môi trường Colab
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
from pathlib import Path
import shutil
import subprocess
import sys

THU_MUC_COLAB_DRIVE = Path("/content/drive/MyDrive/Genetic_ALO_Colab")
THU_MUC_COLAB_DRIVE.mkdir(parents=True, exist_ok=True)
THU_MUC_DU_AN = Path("/content/genetic-alo")
THU_MUC_KET_QUA_DRIVE = THU_MUC_COLAB_DRIVE / "latest_outputs"
REPOSITORY_URL = "https://github.com/duktrung05/genetic-alo.git"

shutil.rmtree(THU_MUC_DU_AN, ignore_errors=True)
subprocess.run(
    [
        "git", "clone", "--depth", "1", "--branch", "main",
        REPOSITORY_URL, str(THU_MUC_DU_AN),
    ],
    check=True,
)

# Khôi phục output mới nhất từ Drive nếu notebook trước đã tạo kết quả.
if THU_MUC_KET_QUA_DRIVE.is_dir():
    shutil.copytree(
        THU_MUC_KET_QUA_DRIVE,
        THU_MUC_DU_AN / "outputs",
        dirs_exist_ok=True,
    )

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--quiet",
        "--disable-pip-version-check", "-r",
        str(THU_MUC_DU_AN / "requirements.txt"),
    ],
    check=True,
)

os.chdir(THU_MUC_DU_AN)
if str(THU_MUC_DU_AN) not in sys.path:
    sys.path.insert(0, str(THU_MUC_DU_AN))

def dong_bo_ket_qua() -> Path:
    """Copy all current outputs to Drive so another notebook can reuse them."""
    THU_MUC_KET_QUA_DRIVE.mkdir(parents=True, exist_ok=True)
    shutil.copytree(
        THU_MUC_DU_AN / "outputs",
        THU_MUC_KET_QUA_DRIVE,
        dirs_exist_ok=True,
    )
    return THU_MUC_KET_QUA_DRIVE

print(f"✅ Đã clone dự án tại: {THU_MUC_DU_AN}")
print(f"✅ Python: {sys.version.split()[0]}")
subprocess.run(["git", "log", "-1", "--oneline"], cwd=THU_MUC_DU_AN, check=True)
print("✅ Dataset Excel nằm trong data/instances.")

In [ ]:
#@title Kiểm tra dữ liệu dùng cho demo
cac_tep_demo = [
    THU_MUC_DU_AN / "ui_app.py",
    THU_MUC_DU_AN / "data/instances/instance_easy.xlsx",
    THU_MUC_DU_AN / "outputs/production/schedule_query_data.json",
    THU_MUC_DU_AN / "outputs/production/best_timetable_metadata.json",
]
tep_thieu = [str(path) for path in cac_tep_demo if not path.is_file()]
if tep_thieu:
    raise FileNotFoundError(
        "Thiếu dữ liệu demo. Hãy chạy notebook 02 trước:\n- " + "\n- ".join(tep_thieu)
    )
print("✅ Dữ liệu demo đã sẵn sàng.")

In [ ]:
#@title Khởi động Streamlit và tạo đường dẫn demo
MO_GIAO_DIEN_STREAMLIT = True #@param {type:"boolean"}

import re
import time
import urllib.request
from IPython.display import HTML, display

if not MO_GIAO_DIEN_STREAMLIT:
    print("ℹ️ Đã bỏ qua việc mở giao diện.")
else:
    # Dừng đúng các tiến trình do notebook này tạo nếu chạy lại cell.
    for ten_bien in ("tien_trinh_streamlit", "tien_trinh_tunnel"):
        tien_trinh_cu = globals().get(ten_bien)
        if tien_trinh_cu is not None and tien_trinh_cu.poll() is None:
            tien_trinh_cu.terminate()

    tep_log_streamlit = Path("/content/streamlit_colab.log")
    tep_log_tunnel = Path("/content/cloudflare_tunnel.log")
    log_streamlit = open(tep_log_streamlit, "w", encoding="utf-8")

    tien_trinh_streamlit = subprocess.Popen(
        [
            sys.executable, "-m", "streamlit", "run", "ui_app.py",
            "--server.headless=true",
            "--server.address=0.0.0.0",
            "--server.port=8501",
            "--browser.gatherUsageStats=false",
        ],
        cwd=THU_MUC_DU_AN,
        env={**os.environ, "GA_DEMO_EVALUATION_BUDGET": "100"},
        stdout=log_streamlit,
        stderr=subprocess.STDOUT,
        text=True,
    )

    # Chờ endpoint health thay vì ngủ một khoảng cố định.
    for _ in range(60):
        if tien_trinh_streamlit.poll() is not None:
            break
        try:
            with urllib.request.urlopen(
                "http://127.0.0.1:8501/_stcore/health", timeout=2
            ) as response:
                if response.status == 200 and response.read().decode().strip() == "ok":
                    break
        except Exception:
            time.sleep(1)
    else:
        raise TimeoutError("Streamlit không phản hồi health check sau 60 giây.")

    if tien_trinh_streamlit.poll() is not None:
        log_streamlit.flush()
        raise RuntimeError(tep_log_streamlit.read_text(errors="replace"))

    url_cong_khai = None
    for lan_thu in range(1, 3):
        log_tunnel = open(tep_log_tunnel, "w", encoding="utf-8")
        tien_trinh_tunnel = subprocess.Popen(
            [
                "npx", "--yes", "wrangler@latest", "tunnel", "quick-start",
                "http://127.0.0.1:8501",
            ],
            stdout=log_tunnel,
            stderr=subprocess.STDOUT,
            text=True,
        )
        for _ in range(90):
            time.sleep(1)
            log_tunnel.flush()
            noi_dung = tep_log_tunnel.read_text(encoding="utf-8", errors="replace")
            ket_qua = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", noi_dung)
            if ket_qua:
                url_cong_khai = ket_qua.group(0)
                break
            if tien_trinh_tunnel.poll() is not None:
                break
        if url_cong_khai:
            break
        if tien_trinh_tunnel.poll() is None:
            tien_trinh_tunnel.terminate()
        print(f"Thử tạo tunnel lần {lan_thu} chưa thành công; đang thử lại...")

    if not url_cong_khai:
        print(tep_log_tunnel.read_text(encoding="utf-8", errors="replace"))
        raise RuntimeError("Không tạo được link demo sau 2 lần thử.")

    print("✅ Streamlit health check: OK")
    print("✅ Link demo:", url_cong_khai)
    display(HTML(f'<a href="{url_cong_khai}" target="_blank" '
                 'style="font-size:20px;font-weight:bold">MỞ WEB DEMO</a>'))

## Phương án dự phòng khi mạng tunnel không ổn định

Notebook 02 vẫn chạy thuật toán, hiển thị bảng thời khóa biểu và biểu đồ ngay
trong Colab. Khi thi, hãy mở sẵn cả notebook 02 và 05; nếu link tạm thời gặp lỗi,
trình bày kết quả trực tiếp từ notebook 02.